# SearchScope — Dense Encoding on GPU (Colab)

Encodes the MS MARCO passage corpus with `bge-base-en-v1.5` on a free Colab GPU instead of your local CPU. Encoding 8.8M passages on CPU can take hours and heats a laptop hard; on a T4 GPU this typically finishes in minutes.

**Steps:**
1. Runtime -> Change runtime type -> select **T4 GPU** (or any available GPU) before running anything below.
2. Get the corpus into Colab — **Option A (recommended) downloads it directly inside Colab**, so you never upload 3GB from your own connection. Options B/C are for if you already have the file elsewhere.
3. Run all cells.
4. Download the two output files (`dense_flat.faiss` and `dense_flat.docids.json`) and drop them into your local repo's `searchscope/indexes/` folder — `retrieval/dense.py`'s `load()` method reads exactly this pair.

In [ ]:
# Confirm you actually got a GPU runtime — if this prints "None", go back
# to Runtime -> Change runtime type -> GPU before continuing.
import torch
print("GPU available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
!pip install -q sentence-transformers==3.0.1 faiss-cpu==1.8.0

## Option A (recommended) — Download the corpus directly inside Colab

Colab's connection to Azure blob storage is typically much faster than uploading 2.9GB from your own machine. This downloads `collection.tar.gz`, extracts it, and reads `collection.tsv` (the same official file `data/download_msmarco.py` expects locally) directly — no local upload needed at all.

In [ ]:
!wget -q --show-progress -O collection.tar.gz https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz
!tar -xzf collection.tar.gz
!ls -lh collection.tsv

CORPUS_PATH = "collection.tsv"
CORPUS_FORMAT = "tsv"  # pid<TAB>text, no header

## Option B — Upload corpus.jsonl from your local machine instead

Skip this if you used Option A above. Slower (uploads through your own connection), but useful if you've already filtered/modified the corpus locally.

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # select corpus.jsonl from your local machine
# CORPUS_PATH = list(uploaded.keys())[0]
# CORPUS_FORMAT = "jsonl"  # {"doc_id": ..., "text": ...} per line

## Option C — Read from Google Drive instead

Useful if you've already uploaded the corpus to Drive once and want to reuse it across sessions without re-downloading.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# CORPUS_PATH = "/content/drive/MyDrive/searchscope/corpus.jsonl"  # adjust to your path
# CORPUS_FORMAT = "jsonl"

## Load the corpus (optionally sample for a quick test run first)

Handles both formats set above — `tsv` (raw MS MARCO `collection.tsv`, from Option A) and `jsonl` (from Options B/C).

In [ ]:
import json
import random

# Set to None to encode the full corpus. Set to e.g. 500_000 to sanity-check
# the pipeline quickly before committing GPU time to the full 8.8M passages.
SAMPLE_SIZE = None
RANDOM_SEED = 42  # matches config.RANDOM_SEED in the main repo

docs = []
with open(CORPUS_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.rstrip("\n")
        if not line:
            continue
        if CORPUS_FORMAT == "tsv":
            pid, text = line.split("\t", 1)
            docs.append({"doc_id": pid, "text": text})
        else:
            docs.append(json.loads(line))
print(f"Loaded {len(docs)} passages")

if SAMPLE_SIZE:
    random.seed(RANDOM_SEED)
    docs = random.sample(docs, SAMPLE_SIZE)
    print(f"Sampled down to {len(docs)} passages")

## Encode on GPU

This mirrors `retrieval/dense.py`'s `DenseRetriever.build_index()` exactly (same model, same `normalize_embeddings=True` so inner product equals cosine similarity), just pointed at the GPU.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

MODEL_NAME = "BAAI/bge-small-en-v1.5"  # matches config.DENSE_ENCODER_NAME (see DESIGN.md §5 for the bge-base -> bge-small tradeoff)

model = SentenceTransformer(MODEL_NAME, device="cuda")
texts = [d["text"] for d in docs]
doc_ids = [d["doc_id"] for d in docs]

start = time.time()
embeddings = model.encode(
    texts,
    batch_size=256,  # GPU can handle a much bigger batch than the CPU default of 64
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.asarray(embeddings, dtype="float32")
print(f"Encoded {len(texts)} passages in {time.time() - start:.1f}s")

## Build the FAISS index and save both output files

Index building itself is CPU work, but it's fast (seconds, not hours) for a flat index — the encoding step above was the actual bottleneck.

In [ ]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

faiss.write_index(index, "dense_flat.faiss")
with open("dense_flat.docids.json", "w") as f:
    json.dump(doc_ids, f)

print("Saved dense_flat.faiss and dense_flat.docids.json")

## Download the results

Drop both files into your local repo at `searchscope/indexes/` (same directory, same filenames) — `config.FAISS_INDEX_PATH` and `DenseRetriever.load()` expect them there together.

These two files are small (the FAISS flat index for 8.8M x 768-dim float32 vectors is a few GB, but that's a normal download, not an upload through your own connection — Colab's `files.download()` is typically fast).

In [ ]:
from google.colab import files

files.download("dense_flat.faiss")
files.download("dense_flat.docids.json")

### Alternative: save to Google Drive instead of downloading

If the FAISS file is large and browser download is slow/flaky, save to Drive instead and copy it down separately (e.g. via `rclone`, Drive's desktop sync app, or the Drive web UI).

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy("dense_flat.faiss", "/content/drive/MyDrive/searchscope/dense_flat.faiss")
# shutil.copy("dense_flat.docids.json", "/content/drive/MyDrive/searchscope/dense_flat.docids.json")

### Back in your local repo

```powershell
# Place the two downloaded files here:
searchscope\indexes\dense_flat.faiss
searchscope\indexes\dense_flat.docids.json

# Then run the baseline eval — it will detect the existing FAISS index
# and load it instead of re-encoding on CPU:
python -m eval.run_baseline_eval --dataset trec-dl-2019
```

You still need the BM25 (Pyserini/Lucene) index built locally via `python -m data.preprocess --dataset trec-dl-2019` — that step is CPU-bound but not the slow part; the dense encoding was.